In [1]:
import numpy as np
import pandas as pd
import pickle as pk
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Conv2D, Dense, Input, MaxPooling2D, Dropout, BatchNormalization, GlobalAveragePooling2D
from sklearn.preprocessing import LabelEncoder

In [2]:
import kagglehub

path = kagglehub.dataset_download("zlatan599/mushroom1")

print("Path to dataset files:", path)

100%|██████████| 11.3G/11.3G [06:24<00:00, 31.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/zlatan599/mushroom1/versions/2


In [4]:
train_df = pd.read_csv('/root/.cache/kagglehub/datasets/zlatan599/mushroom1/versions/2/train.csv')
test_df = pd.read_csv('/root/.cache/kagglehub/datasets/zlatan599/mushroom1/versions/2/test.csv')
val_df = pd.read_csv('/root/.cache/kagglehub/datasets/zlatan599/mushroom1/versions/2/val.csv')

Le = LabelEncoder()
train_df["encode_label"] = Le.fit_transform(train_df["label"])
val_df["encode_label"] = Le.transform(val_df["label"])
test_df["encode_label"] = Le.transform(test_df["label"])

In [6]:
img_size = (64, 64)
batch_size = 32

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/root/.cache/kagglehub/datasets/zlatan599/mushroom1/versions/2/merged_dataset",
    labels='inferred',
    label_mode='int',
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/root/.cache/kagglehub/datasets/zlatan599/mushroom1/versions/2/merged_dataset",
    labels='inferred',
    label_mode='int',
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

autotune = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(autotune )
validation_dataset  = validation_dataset.prefetch(autotune)

Found 104088 files belonging to 169 classes.
Using 83271 files for training.
Found 104088 files belonging to 169 classes.
Using 20817 files for validation.


In [ ]:
model = Sequential()

model.add(Input(shape=(64,64,3)))
model.add(Conv2D(32,kernel_size=(3,3),activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(64,kernel_size=(3,3),activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(128,kernel_size=(3,3),activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(256,kernel_size=(3,3),activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(GlobalAveragePooling2D())

model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(169,activation='softmax'))

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
    )

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    batch_size=32,
    verbose=1
)

In [ ]:
model.save('modelo_imagenes.h5')

with open('label_encoder.pkl', 'wb') as archivo:
    pk.dump(Le, archivo)

paquete_completo = {
    'label_encoder': Le,
    'clases': Le.classes_,
    'num_clases': len(Le.classes_),
    'input_shape': (64, 64, 3)
}

with open('metadata_modelo.pkl', 'wb') as archivo:
    pk.dump(paquete_completo, archivo)